# 実行環境：Python と設定ファイルの検証

対応する本文: [setup.tex](../chapters/setup.tex)。各問を「考え方 → 実装 → 確認」の順で解説する。上から全セルを実行すると、入力の作成から結果の検証までを再現できる。生成物は専用の出力先へ保存する。

In [1]:
import sys
from pathlib import Path

# リポジトリ直下と notebooks/ のどちらから起動しても同じ場所を使う。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "main.tex").exists() and (p / "chapters").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("教材のフォルダー内で Notebook を起動してください。")
WORK = ROOT / "notebooks" / "_outputs" / "02_setup"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("出力先:", WORK.relative_to(ROOT))

Python 3.12.14
出力先: notebooks/_outputs/02_setup


## 問1：uv のプロジェクトを確認する

本文のアプリケーション例とそろえる場合は `--package` を明示する。uv 0.12 より前と以降では既定構成が異なるが、この指定なら `src/` とビルド設定を持つ構成になる。新しいプロジェクトで次を実行する。

```sh
uv init --package usgs_project
cd usgs_project
uv add numpy matplotlib scipy
uv run python --version
uv run python -c "import sys, numpy, matplotlib, scipy; print(sys.executable)"
```

`uv add` は依存関係を記録して環境を同期する。`.python-version` は Python の選択、`pyproject.toml` は Python の対応範囲と依存関係、`uv.lock` は解決結果を記録する。`-c` は文字列を Python コードとして実行する。

Notebook は導入済みの環境で計算する。次のセルでは実際の実行ファイルとライブラリを確認する。

In [2]:
from importlib import import_module

print("実行ファイル:", sys.executable)
print("仮想環境:", sys.prefix != sys.base_prefix)
for name in ["numpy", "matplotlib", "scipy"]:
    print(name, import_module(name).__version__)

実行ファイル: /Users/ikomae/src/python_test/tex/.venv/bin/python
仮想環境: True
numpy 2.5.4
matplotlib 3.11.2
scipy 1.18.1


## 問2：venv の Python を直接起動する

`python -m venv .venv` の `-m` はモジュールの実行、末尾の `.venv` は作成先を表す。有効化は現在のシェルの PATH を変更する操作である。Notebook の `!source ...` では以降のセルの Python は変わらない。カーネルを切り替えるか、仮想環境の実行ファイルを明示する。

次は実習用の小さな venv を作り、その Python を直接起動する。pip と追加ライブラリはこの実習用 venv には導入しない。

端末では作成前後に `command -v python`、`python --version`、`python -m pip --version` を記録し、`source .venv/bin/activate` と `deactivate` による変化を調べる。

In [3]:
import json
import subprocess
import venv

ENV_DIR = WORK / "venv_example"
venv.EnvBuilder(with_pip=False).create(ENV_DIR)
python_in_venv = ENV_DIR / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
probe = "import sys,json; print(json.dumps({'executable':sys.executable,'prefix':sys.prefix,'base_prefix':sys.base_prefix}))"
result = subprocess.run([str(python_in_venv), "-c", probe], capture_output=True, text=True, check=True)
info = json.loads(result.stdout)
assert Path(info["prefix"]).resolve() == ENV_DIR.resolve()
assert info["prefix"] != info["base_prefix"]
print("作成した venv から起動できました。")

作成した venv から起動できました。


## 問3：pyenv の shim と実体を区別する

`command -v python` が示す shim は中継用の実行ファイルである。`pyenv which python` は pyenv が選んだ実体を示す。2系統の Python を導入済みの端末で、別々のフォルダーに移動して次を比較する。使用する版は `pyenv versions` にあるものを選ぶ。

```sh
pyenv local 3.11.11
python --version
command -v python
pyenv which python
cat .python-version
python -c "import sys; print(sys.executable); print(sys.version)"
```

もう一方で別の版を設定する。`uv run python --version` はプロジェクト環境の選択なので、shim の表示だけで一致を判断しない。Notebook の実行だけでは親の対話シェルの有効化や pyenv の切り替えは検証できない。

## 補足：設定値を読み戻す

時間窓の単位と型も検査する。JSON の `true` は Python で `True` となり、`bool` は `int` の派生型なので、整数だけを受け付ける場合は型の条件を明示する。

In [4]:
config_path = WORK / "config.json"
config_path.write_text(json.dumps({"coincidence_window_ns": 250, "figure_format": "pdf"}, indent=2), encoding="utf-8")
config = json.loads(config_path.read_text(encoding="utf-8"))
window_ns = config["coincidence_window_ns"]
assert type(window_ns) is int and window_ns >= 0
print(config)

{'coincidence_window_ns': 250, 'figure_format': 'pdf'}
